# 02 — Monte Carlo variance reduction

Builds three Monte Carlo engines via `MonteCarloEngineBuilder`:
1. Plain
2. With `AntitheticDecorator`
3. With both Antithetic and `ControlVariateDecorator`

Prices a European call: $\hat C = \frac{e^{-rT}}{N}\sum_{i=1}^N \max(S_T^{(i)}-K, 0)$.

**Prerequisites**: `MonteCarloEngineBuilder::build()`, both decorators, plus a concrete `IMCEstimator` implementation that you'll write.

In [ ]:
import sys
from pathlib import Path
for so in Path('../build').rglob('nmpy*.so'):
    sys.path.insert(0, str(so.parent))
    break
import nmpy
import numpy as np
import matplotlib.pyplot as plt
import math

In [ ]:
S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.20, 1.0
drift = (r - 0.5*sigma**2)*T
vol   = sigma*math.sqrt(T)

def call_payoff(z):
    """z is a draw from N(0,1); returns discounted payoff."""
    S_T = S0 * math.exp(drift + vol*z)
    return math.exp(-r*T) * max(S_T - K, 0.0)

def stock_price_cv(z):
    return S0 * math.exp(drift + vol*z)

cv_mean = S0 * math.exp(r*T)   # E[S_T] under risk-neutral measure

In [ ]:
def build(plain=False, av=False, cv=False):
    b = nmpy.MonteCarloEngineBuilder().with_payoff(call_payoff).with_seed(42)
    if av: b = b.with_antithetic()
    if cv: b = b.with_control_variate(stock_price_cv, cv_mean)
    return b

Ns = [int(n) for n in np.logspace(2, 5, 8)]
results = {'plain': [], 'antithetic': [], 'antithetic+CV': []}
for N in Ns:
    results['plain'].append(build().with_paths(N).build().estimate(N, 42).stderr)
    results['antithetic'].append(build(av=True).with_paths(N).build().estimate(N, 42).stderr)
    results['antithetic+CV'].append(build(av=True, cv=True).with_paths(N).build().estimate(N, 42).stderr)

for label, ses in results.items():
    plt.loglog(Ns, ses, label=label)
plt.xlabel('paths N'); plt.ylabel('stderr'); plt.title('Variance reduction'); plt.legend(); plt.show()